In [ ]:
# -*- coding: utf-8 -*-
"""PlantDiseaseCP1.ipynb"""

# --- General Imports ---
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
# Install kaggle CLI
!pip install -q kaggle

# Upload your kaggle.json API token here, or use opendatasets
import kagglehub
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("--- Step 1: Loading Dataset ---")
data_base_dir = "/kaggle/input/plantvillage-dataset/plantvillage dataset/color"

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
--- Step 1: Loading Dataset ---


In [ ]:


"""# Epic 1: Data Engineering
## Story 1: Data Collection and Preparation
### Step 1: Load and Augment Dataset"""

print("--- Step 1: Loading Dataset ---")
data_base_dir = "/kaggle/input/plantvillage-dataset/plantvillage dataset/color"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2 # Use 20% of data for validation
)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2 # Use 20% of data for validation
)

train_gen = train_datagen.flow_from_directory(
    data_base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset='training', # Specify training subset
    seed=42 # For reproducibility
)
val_gen = val_datagen.flow_from_directory(
    data_base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset='validation', # Specify validation subset
    seed=42 # For reproducibility
)

"""# Epic 2: Model Development
## Story 1: Transfer Learning Model (MobileNetV2)"""

base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_gen, validation_data=val_gen, epochs=13)


# Plotting training accuracy and loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


"""# Epic 3: Model Optimization
## Story 1: Convert to TensorFlow Lite"""

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open("plant_disease_model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)
print("Quantized TFLite model saved.")


interpreter = tf.lite.Interpreter(model_path="plant_disease_model_quant.tflite")
interpreter.allocate_tensors()
print("TFLite model ready for real-time inference deployment.")


# Keras vs TFLite model optimization comparison
import os

keras_model_path = "plant_disease_model_keras.h5"
model.save(keras_model_path) # Save the Keras model for size comparison

keras_model_size = os.path.getsize(keras_model_path) / (1024 * 1024) # Size in MB
tflite_model_size = os.path.getsize("plant_disease_model_quant.tflite") / (1024 * 1024) # Size in MB

print("\n--- Keras vs TFLite Model Optimization Comparison ---")
print(f"Keras Model Size: {keras_model_size:.2f} MB")
print(f"TFLite Quantized Model Size: {tflite_model_size:.2f} MB")
print(f"Size Reduction: {((keras_model_size - tflite_model_size) / keras_model_size * 100):.2f}%")

print("\nOptimization Benefit: The TFLite model uses 8-bit integer quantization, which significantly reduces the model size and can lead to faster inference speeds and lower memory footprint on edge devices, often with minimal loss in accuracy.")

print("\nKeras Model Summary (Parameters):")
model.summary()

print("\nTFLite Interpreter Input/Output Details:")
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input Details:", input_details)
print("Output Details:", output_details)


--- Step 1: Loading Dataset ---
Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 891s 642ms/step - accuracy: 0.8017 - loss: 0.6718 - val_accuracy: 0.9072 - val_loss: 0.2863
Epoch 2/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 661s 486ms/step - accuracy: 0.8914 - loss: 0.3319 - val_accuracy: 0.9242 - val_loss: 0.2313
Epoch 3/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 631s 465ms/step - accuracy: 0.9042 - loss: 0.2914 - val_accuracy: 0.9254 - val_loss: 0.2246
Epoch 4/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 640s 471ms/step - accuracy: 0.9134 - loss: 0.2637 - val_accuracy: 0.9294 - val_loss: 0.2091
Epoch 5/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 647s 476ms/step - accuracy: 0.9177 - loss: 0.2471 - val_accuracy: 0.9385 - val_loss: 0.1876
Epoch 6/13
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 651s 479ms/step - accuracy: 0.9229 - loss: 0.2307 - val_accuracy: 0.9343 - val_loss: 0.1980
Epoch 7/13
1358/1358 ━━━━━━

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
